## Part 04 Performance Arena

**Objectives** : Chunking Strategy Comparison, CAG Cache Effectiveness, CRAG Correction Impact, Cost Analysis

### 1. Chunking Strategy Comparison

Evaluates 10 queries across 5 strategies. Must include Precision@5, Recall@5, Answer Relevance, and Latency metrics Identifies clear winner.

### 2. CAG Cache Effectiveness

Simulates 100 queries. Calculates cache hit rate, latency improvement, and estimated cost savings from avoided API calls.

### 3. CRAG Correction Impact

Compares RAG vs CRAG on 20 queries. Tracks correction frequency, confidence improvement, and answer quality gains.

### 4. Cost Analysis

Detailed monthly cost breakdown (API + storage) and ROI analysis. Must use realistic assumptions for scale (e.g., 500 daily users)


In [10]:
import os
import sys
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment
load_dotenv(project_root / ".env")

# Check for API key (OpenRouter preferred, OpenAI as fallback)
openrouter_key = os.getenv("OPENROUTER_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not openrouter_key and not openai_key:
    raise EnvironmentError(
        "❌ No API key found!\n"
        "   Add OPENROUTER_API_KEY (recommended) or OPENAI_API_KEY to .env"
    )

# Load configuration
from real_state.config import (
    VECTOR_DIR, CACHE_DIR, TOP_K_RESULTS,
    CHAT_MODEL, EMBEDDING_MODEL, PROVIDER
)

provider = "OpenRouter" if openrouter_key else "OpenAI"
print("✅ Environment loaded")

✅ Environment loaded


#### Test Queries & Ground Truth

In [11]:
test_queries = [
    # 1. Direct price lookup (simple, should be confident)
    {
        "query": "What is the price of MAJESTINO MORATUWA?",
        "type": "Price Lookup",
        "expected": "1,050,000 LKR",
        "relevant_urls": ["https://www.primelands.lk/land/MAJESTINO-MORATUWA/en"]
    },
    # 2. Specific project details (precise, single-project)
    {
        "query": "What amenities and facilities are available at GREEN EMBAZY HOMAGAMA?",
        "type": "Feature Query",
        "expected": "Facilities list from project ID 184",
        "relevant_urls": ["https://www.primelands.lk/land/GREEN-EMBAZY-HOMAGAMA/en"]
    },
    # 3. Payment plan query
    {
        "query": "What is the payment plan for EVARA NAWALAPITIYA?",
        "type": "Payment Plan",
        "expected": "Down payment + installment details",
        "relevant_urls": ["https://www.primelands.lk/land/EVARA-NAWALAPITIYA/en"]
    },
    # 4. Location-based query (multi-project)
    {
        "query": "Which PrimeLands projects are available in Kadawatha?",
        "type": "Location Filter",
        "expected": "Multiple Kadawatha projects",
        "relevant_urls": [
            "https://www.primelands.lk/land/CALLISTO-KADAWATHA/en",
            "https://www.primelands.lk/land/EXPRESS-CORRIDOR-KADAWATHA/en",
            "https://www.primelands.lk/land/PREMIER-KADAWATHA/en",
            "https://www.primelands.lk/land/360-AVENUE-KADAWATHA/en",
            "https://www.primelands.lk/land/PRIME-ATRIUM-KADAWATHA/en"
        ]
    },
    # 5. Comparison query (complex)
    {
        "query": "Compare apartment prices in Colombo vs Negombo",
        "type": "Comparison",
        "expected": "THE SEASONS 127M vs VENEZIA, J ADORE in Negombo",
        "relevant_urls": [
            "https://www.primelands.lk/apartment/THE-SEASONS-COLOMBO-08/en",
            "https://www.primelands.lk/apartment/MON-VIE-COLOMBO-05/en",
            "https://www.primelands.lk/apartment/VENEZIA-NEGOMBO/en",
            "https://www.primelands.lk/apartment/J-ADORE-NEGOMBO/en"
        ]
    },
    # 6. Vague/broad query
    {
        "query": "Tell me about land projects in the south",
        "type": "Vague Regional",
        "expected": "Southern projects: Ahangama, Hikkaduwa, Habaraduwa",
        "relevant_urls": [
            "https://www.primelands.lk/land/GLORIA-STAGE-II-AHANGAMA/en",
            "https://www.primelands.lk/land/HABARADUWA-BLUE-SHORE/en",
            "https://www.primelands.lk/land/MATARA-THE-KINGDOM/en",
            "https://www.primelands.lk/land/GREATE-10-GALLE/en"
        ]
    },
    # 7. FAQ-style query
    {
        "query": "How much does SKYE BLOSSOM KOTTAWA cost?",
        "type": "FAQ Match",
        "expected": "50,000,000 LKR",
        "relevant_urls": ["https://www.primelands.lk/apartment/SKYE-BLOSSOM-KOTTAWA/en"]
    },
    # 8. Proximity/accessibility query (multi-part)
    {
        "query": "Which projects are close to railway stations and what are their prices?",
        "type": "Multi-Part",
        "expected": "Projects mentioning railway station proximity",
        "relevant_urls": [
            "https://www.primelands.lk/land/MAJESTINO-MORATUWA/en",
            "https://www.primelands.lk/land/ZILLION-MORATUWA/en"
        ]
    },
    # 9. House-specific query
    {
        "query": "What houses are available in Thalawathugoda and what are the features?",
        "type": "Category + Location",
        "expected": "CEST LA VIE, CLOVER, THE RESIDENCE",
        "relevant_urls": [
            "https://www.primelands.lk/house/CEST-LA-VIE-THALAWATHUGODA/en",
            "https://www.primelands.lk/house/THALAWATHUGODA-CLOVER/en",
            "https://www.primelands.lk/house/THE-RESIDENCE-SAMAGI-MAWATHA-THALAWATHUGODA/en"
        ]
    },
    # 10. Investment query (abstract)
    {
        "query": "Which PrimeLands property has the lowest price per perch for investment?",
        "type": "Abstract/Investment",
        "expected": "Projects with lowest per-perch prices",
        "relevant_urls": [
            "https://www.primelands.lk/land/GLORIA-STAGE-II-AHANGAMA/en",
            "https://www.primelands.lk/land/PRIME-KESHARA-ADAWIYA/en"
        ]
    }
]

print(f"✅ {len(test_queries)} test queries loaded")
for i, q in enumerate(test_queries, 1):
    print(f"   {i:2d}. [{q['type']:22s}] {q['query'][:60]}")

✅ 10 test queries loaded
    1. [Price Lookup          ] What is the price of MAJESTINO MORATUWA?
    2. [Feature Query         ] What amenities and facilities are available at GREEN EMBAZY 
    3. [Payment Plan          ] What is the payment plan for EVARA NAWALAPITIYA?
    4. [Location Filter       ] Which PrimeLands projects are available in Kadawatha?
    5. [Comparison            ] Compare apartment prices in Colombo vs Negombo
    6. [Vague Regional        ] Tell me about land projects in the south
    7. [FAQ Match             ] How much does SKYE BLOSSOM KOTTAWA cost?
    8. [Multi-Part            ] Which projects are close to railway stations and what are th
    9. [Category + Location   ] What houses are available in Thalawathugoda and what are the
   10. [Abstract/Investment   ] Which PrimeLands property has the lowest price per perch for


---
## 1️⃣ Chunking Strategy Comparison

Evaluates 10 queries across 5 chunking strategies with **Precision@5**, **Recall@5**, **Answer Relevance**, and **Latency** metrics.

In [13]:
from real_state.infrastructure.db.vector_db import QdrantStorage
from real_state.infrastructure.llm_provider import get_default_embeddings, get_chat_llm
from real_state.application.chat_service import RagService
from real_state.application.domain.utils import format_docs

emb = get_default_embeddings()
llm = get_chat_llm(temperature=0)
db_client = QdrantStorage()

info = db_client.collection_info(collection_name="prime_lands")
print(f"✅ Qdrant connected: {info['points_count']} vectors, {info['distance']}")
print(f"✅ LLM: {CHAT_MODEL}")
print(f"✅ Embeddings: {EMBEDDING_MODEL}")

✅ Qdrant connected: 2015 vectors, COSINE
✅ LLM: google/gemini-2.5-flash
✅ Embeddings: openai/text-embedding-3-large


In [14]:
# ============================================================
# Evaluation Metrics
# ============================================================

STRATEGIES = ["semantic", "fixed", "sliding", "child", "late_chunk_base"]
K = 5

def precision_at_k(retrieved_urls: list, relevant_urls: list, k: int = K) -> float:
    """Fraction of top-k retrieved docs that are relevant."""
    top_k = retrieved_urls[:k]
    if not top_k:
        return 0.0
    hits = sum(1 for url in top_k if url in relevant_urls)
    return hits / len(top_k)


def recall_at_k(retrieved_urls: list, relevant_urls: list, k: int = K) -> float:
    """Fraction of relevant docs that appear in top-k."""
    if not relevant_urls:
        return 0.0
    top_k = retrieved_urls[:k]
    hits = sum(1 for url in relevant_urls if url in top_k)
    return hits / len(relevant_urls)


def answer_relevance(answer: str, expected: str, query: str) -> float:
    """
    Compute answer relevance score (0-1) based on:
    - Keyword overlap with expected answer
    - Query term coverage in the answer
    - Penalty for 'not available' / 'no information' responses
    """
    answer_lower = answer.lower()
    
    # Penalty for refusal / no-answer
    refusal_phrases = ["not available", "no information", "cannot answer", "don't have"]
    if any(p in answer_lower for p in refusal_phrases):
        return 0.1
    
    # 1. Expected keyword overlap (weight: 50%)
    expected_words = set(expected.lower().split())
    answer_words = set(answer_lower.split())
    if expected_words:
        keyword_score = len(expected_words & answer_words) / len(expected_words)
    else:
        keyword_score = 0.0
    
    # 2. Query term presence in answer (weight: 30%)
    query_words = set(query.lower().split()) - {"what", "is", "the", "of", "are", "in", "and", "for", "a", "an"}
    if query_words:
        query_score = len(query_words & answer_words) / len(query_words)
    else:
        query_score = 0.0
    
    # 3. Answer length quality (weight: 20%) — too short or empty penalised
    word_count = len(answer.split())
    length_score = min(word_count / 50, 1.0)
    
    return 0.5 * keyword_score + 0.3 * query_score + 0.2 * length_score


print("✅ Evaluation functions defined")
print(f"📊 Strategies: {STRATEGIES}")
print(f"📊 K = {K}")

✅ Evaluation functions defined
📊 Strategies: ['semantic', 'fixed', 'sliding', 'child', 'late_chunk_base']
📊 K = 5


In [16]:
# ============================================================
# Run Evaluation: 10 queries × 5 strategies
# ============================================================

from real_state.application.chat_service.rag_service import build_rag_chain

chain = build_rag_chain(llm)

results = []

total_runs = len(test_queries) * len(STRATEGIES)
run_count = 0

print("=" * 80)
print("🏟️  PERFORMANCE ARENA: 10 Queries × 5 Strategies")
print("=" * 80)

for qi, tq in enumerate(test_queries, 1):
    query = tq["query"]
    relevant_urls = tq["relevant_urls"]
    expected = tq["expected"]
    qtype = tq["type"]
    
    print(f"\n📝 Query {qi}/10: {query[:60]}...")
    
    for strategy in STRATEGIES:
        run_count += 1
        print(f"   [{run_count:2d}/{total_runs}] Strategy: {strategy:18s}", end="", flush=True)
        
        try:
            # --- Retrieve ---
            t_start = time.time()
            docs = db_client.search_chunks(
                query=query,
                top_k=K,
                strategy_filter=strategy
            )
            t_retrieve = time.time() - t_start
            
            retrieved_urls = [d.get("url", "") for d in docs]
            
            # --- Generate answer ---
            t_start = time.time()
            context = format_docs(docs)
            answer = chain.invoke({"context": context, "question": query})
            t_generate = time.time() - t_start
            
            t_total = t_retrieve + t_generate
            
            # --- Compute metrics ---
            p_at_k = precision_at_k(retrieved_urls, relevant_urls, K)
            r_at_k = recall_at_k(retrieved_urls, relevant_urls, K)
            a_rel = answer_relevance(answer, expected, query)
            
            avg_score = np.mean([d.get("score", 0) for d in docs]) if docs else 0.0
            
            results.append({
                "query_id": qi,
                "query_type": qtype,
                "query": query,
                "strategy": strategy,
                "precision_at_5": p_at_k,
                "recall_at_5": r_at_k,
                "answer_relevance": a_rel,
                "latency_s": round(t_total, 2),
                "retrieve_s": round(t_retrieve, 2),
                "generate_s": round(t_generate, 2),
                "avg_score": round(avg_score, 4),
                "docs_returned": len(docs),
                "answer_preview": answer[:150]
            })
            
            print(f"  P@5={p_at_k:.2f}  R@5={r_at_k:.2f}  Rel={a_rel:.2f}  ⏱️{t_total:.1f}s")
            
        except Exception as e:
            print(f"  ❌ Error: {str(e)[:60]}")
            results.append({
                "query_id": qi,
                "query_type": qtype,
                "query": query,
                "strategy": strategy,
                "precision_at_5": 0.0,
                "recall_at_5": 0.0,
                "answer_relevance": 0.0,
                "latency_s": 0.0,
                "retrieve_s": 0.0,
                "generate_s": 0.0,
                "avg_score": 0.0,
                "docs_returned": 0,
                "answer_preview": f"ERROR: {str(e)[:100]}"
            })

print(f"\n✅ Evaluation complete: {len(results)} runs")

🏟️  PERFORMANCE ARENA: 10 Queries × 5 Strategies

📝 Query 1/10: What is the price of MAJESTINO MORATUWA?...
   [ 1/50] Strategy: semantic            P@5=0.20  R@5=1.00  Rel=0.90  ⏱️6.5s
   [ 2/50] Strategy: fixed               P@5=0.20  R@5=1.00  Rel=0.90  ⏱️4.9s
   [ 3/50] Strategy: sliding             P@5=0.40  R@5=1.00  Rel=0.90  ⏱️3.9s
   [ 4/50] Strategy: child               P@5=0.60  R@5=1.00  Rel=0.90  ⏱️4.7s
   [ 5/50] Strategy: late_chunk_base     P@5=0.20  R@5=1.00  Rel=0.90  ⏱️3.8s

📝 Query 2/10: What amenities and facilities are available at GREEN EMBAZY ...
   [ 6/50] Strategy: semantic            P@5=0.20  R@5=1.00  Rel=0.70  ⏱️3.9s
   [ 7/50] Strategy: fixed               P@5=0.40  R@5=1.00  Rel=0.58  ⏱️3.8s
   [ 8/50] Strategy: sliding             P@5=0.60  R@5=1.00  Rel=0.50  ⏱️4.8s
   [ 9/50] Strategy: child               P@5=0.60  R@5=1.00  Rel=0.66  ⏱️4.3s
   [10/50] Strategy: late_chunk_base     P@5=0.20  R@5=1.00  Rel=0.45  ⏱️3.4s

📝 Query 3/10: What is the paymen

In [17]:
# ============================================================
# Results: Per-Strategy Summary Table
# ============================================================

df = pd.DataFrame(results)

summary = df.groupby("strategy").agg(
    Avg_Precision_5=("precision_at_5", "mean"),
    Avg_Recall_5=("recall_at_5", "mean"),
    Avg_Answer_Relevance=("answer_relevance", "mean"),
    Avg_Latency_s=("latency_s", "mean"),
    Avg_Qdrant_Score=("avg_score", "mean"),
    Total_Docs=("docs_returned", "sum")
).round(3)

# Add composite score: weighted sum of P@5, R@5, and Relevance (equal weight)
summary["Composite_Score"] = (
    summary["Avg_Precision_5"] * 0.3 +
    summary["Avg_Recall_5"] * 0.3 +
    summary["Avg_Answer_Relevance"] * 0.4
).round(3)

summary = summary.sort_values("Composite_Score", ascending=False)

print("=" * 90)
print("📊 STRATEGY COMPARISON SUMMARY (sorted by Composite Score)")
print("=" * 90)
print(summary.to_string())
print("\n" + "=" * 90)

📊 STRATEGY COMPARISON SUMMARY (sorted by Composite Score)
                 Avg_Precision_5  Avg_Recall_5  Avg_Answer_Relevance  Avg_Latency_s  Avg_Qdrant_Score  Total_Docs  Composite_Score
strategy                                                                                                                          
child                       0.40         0.623                 0.515          5.421             0.523          50            0.513
fixed                       0.20         0.553                 0.510          4.556             0.494          50            0.430
sliding                     0.26         0.503                 0.501          4.855             0.517          50            0.429
semantic                    0.14         0.478                 0.540          4.514             0.503          50            0.401
late_chunk_base             0.16         0.503                 0.450          4.431             0.494          50            0.379



In [18]:
# ============================================================
# Identify the Winner
# ============================================================

winner = summary.index[0]
runner_up = summary.index[1] if len(summary) > 1 else "N/A"

print("=" * 80)
print("🏆 CHUNKING STRATEGY WINNER")
print("=" * 80)

print(f"\n🥇 Winner: {winner.upper()}")
print(f"   Composite Score: {summary.loc[winner, 'Composite_Score']:.3f}")
print(f"   Precision@5:    {summary.loc[winner, 'Avg_Precision_5']:.3f}")
print(f"   Recall@5:       {summary.loc[winner, 'Avg_Recall_5']:.3f}")
print(f"   Relevance:      {summary.loc[winner, 'Avg_Answer_Relevance']:.3f}")
print(f"   Avg Latency:    {summary.loc[winner, 'Avg_Latency_s']:.2f}s")

print(f"\n🥈 Runner-up: {runner_up.upper()}")
if runner_up != "N/A":
    print(f"   Composite Score: {summary.loc[runner_up, 'Composite_Score']:.3f}")
    margin = summary.loc[winner, 'Composite_Score'] - summary.loc[runner_up, 'Composite_Score']
    print(f"   Margin: +{margin:.3f}")

# Per-query-type breakdown for winner
print(f"\n" + "-" * 80)
print(f"📋 {winner.upper()} — Per Query Type Breakdown")
print("-" * 80)

winner_df = df[df["strategy"] == winner]
for _, row in winner_df.iterrows():
    print(f"   Q{row['query_id']:2d} [{row['query_type']:22s}] "
          f"P@5={row['precision_at_5']:.2f}  R@5={row['recall_at_5']:.2f}  "
          f"Rel={row['answer_relevance']:.2f}  ⏱️{row['latency_s']:.1f}s")

print("\n" + "=" * 80)

🏆 CHUNKING STRATEGY WINNER

🥇 Winner: CHILD
   Composite Score: 0.513
   Precision@5:    0.400
   Recall@5:       0.623
   Relevance:      0.515
   Avg Latency:    5.42s

🥈 Runner-up: FIXED
   Composite Score: 0.430
   Margin: +0.083

--------------------------------------------------------------------------------
📋 CHILD — Per Query Type Breakdown
--------------------------------------------------------------------------------
   Q 1 [Price Lookup          ] P@5=0.60  R@5=1.00  Rel=0.90  ⏱️4.7s
   Q 2 [Feature Query         ] P@5=0.60  R@5=1.00  Rel=0.66  ⏱️4.3s
   Q 3 [Payment Plan          ] P@5=0.40  R@5=1.00  Rel=0.62  ⏱️5.1s
   Q 4 [Location Filter       ] P@5=0.40  R@5=0.40  Rel=0.59  ⏱️6.4s
   Q 5 [Comparison            ] P@5=0.80  R@5=0.50  Rel=0.10  ⏱️5.0s
   Q 6 [Vague Regional        ] P@5=0.00  R@5=0.00  Rel=0.40  ⏱️8.7s
   Q 7 [FAQ Match             ] P@5=0.40  R@5=1.00  Rel=0.83  ⏱️5.3s
   Q 8 [Multi-Part            ] P@5=0.60  R@5=1.00  Rel=0.10  ⏱️4.1s
   Q 9 [Category

In [20]:
# ============================================================
# Detailed Heatmap: Strategy × Query
# ============================================================

print("📊 Precision@5 by Strategy × Query")
print("=" * 80)

pivot_precision = df.pivot_table(
    index="query_id", columns="strategy", values="precision_at_5"
).round(2)

# Add query type labels
query_labels = {q['query_id'] if 'query_id' in q else i+1: q['type'] 
                for i, q in enumerate(test_queries)}
pivot_precision.index = [f"Q{i} ({query_labels.get(i, '')})" for i in pivot_precision.index]

print(pivot_precision.to_string())

print("\n" + "=" * 80)
print("📊 Recall@5 by Strategy × Query")
print("=" * 80)

pivot_recall = df.pivot_table(
    index="query_id", columns="strategy", values="recall_at_5"
).round(2)
pivot_recall.index = [f"Q{i} ({query_labels.get(i, '')})" for i in pivot_recall.index]

print(pivot_recall.to_string())

print("\n" + "=" * 80)
print("📊 Answer Relevance by Strategy × Query")
print("=" * 80)

pivot_relevance = df.pivot_table(
    index="query_id", columns="strategy", values="answer_relevance"
).round(2)
pivot_relevance.index = [f"Q{i} ({query_labels.get(i, '')})" for i in pivot_relevance.index]

print(pivot_relevance.to_string())

print("\n" + "=" * 80)
print("⏱️  Latency (seconds) by Strategy × Query")
print("=" * 80)

pivot_latency = df.pivot_table(
    index="query_id", columns="strategy", values="latency_s"
).round(2)
pivot_latency.index = [f"Q{i} ({query_labels.get(i, '')})" for i in pivot_latency.index]

print(pivot_latency.to_string())

📊 Precision@5 by Strategy × Query
strategy                   child  fixed  late_chunk_base  semantic  sliding
Q1 (Price Lookup)            0.6    0.2              0.2       0.2      0.4
Q2 (Feature Query)           0.6    0.4              0.2       0.2      0.6
Q3 (Payment Plan)            0.4    0.2              0.2       0.2      0.2
Q4 (Location Filter)         0.4    0.2              0.2       0.2      0.2
Q5 (Comparison)              0.8    0.4              0.4       0.2      0.6
Q6 (Vague Regional)          0.0    0.0              0.0       0.0      0.0
Q7 (FAQ Match)               0.4    0.2              0.2       0.2      0.4
Q8 (Multi-Part)              0.6    0.2              0.0       0.0      0.0
Q9 (Category + Location)     0.2    0.2              0.2       0.2      0.2
Q10 (Abstract/Investment)    0.0    0.0              0.0       0.0      0.0

📊 Recall@5 by Strategy × Query
strategy                   child  fixed  late_chunk_base  semantic  sliding
Q1 (Price Lookup)     

In [23]:
# ============================================================
# Save raw results to CSV for further analysis
# ============================================================

from real_state.config import CRAWL_OUT_DIR

csv_path = CRAWL_OUT_DIR / "chunking_comparison_results.csv"
df.to_csv(csv_path, index=False)
print(f"✅ Raw results saved to: {csv_path}")
print(f"   {len(df)} rows × {len(df.columns)} columns")

summary_path = CRAWL_OUT_DIR / "chunking_comparison_summary.csv"
summary.to_csv(summary_path)
print(f"✅ Summary saved to: {summary_path}")

✅ Raw results saved to: d:\ai\bootcamp\mini-project-02\real_state\data\chunking_comparison_results.csv
   50 rows × 13 columns
✅ Summary saved to: d:\ai\bootcamp\mini-project-02\real_state\data\chunking_comparison_summary.csv


In [22]:
# ============================================================
# Final Verdict — Trade-off Analysis
# ============================================================

print("=" * 80)
print("📋 FINAL VERDICT: CHUNKING STRATEGY TRADE-OFFS")
print("=" * 80)

for strat in summary.index:
    row = summary.loc[strat]
    
    # Determine medal
    rank = list(summary.index).index(strat) + 1
    medal = {1: "🥇", 2: "🥈", 3: "🥉"}.get(rank, "  ")
    
    print(f"\n{medal} {strat.upper()} (Rank #{rank})")
    print(f"   Composite: {row['Composite_Score']:.3f} | "
          f"P@5: {row['Avg_Precision_5']:.3f} | "
          f"R@5: {row['Avg_Recall_5']:.3f} | "
          f"Rel: {row['Avg_Answer_Relevance']:.3f} | "
          f"Latency: {row['Avg_Latency_s']:.2f}s")
    
    # Strategy-specific insights
    if strat == "semantic":
        print("   💡 Best for: Topic-specific queries (price lookups, feature queries)")
        print("   ⚠️  Watch: May miss cross-section answers")
    elif strat == "fixed":
        print("   💡 Best for: Predictable, uniform retrieval")
        print("   ⚠️  Watch: Splits mid-section, ignores document structure")
    elif strat == "sliding":
        print("   💡 Best for: Maximum coverage, no content gaps")
        print("   ⚠️  Watch: 50% redundancy, diluted embeddings")
    elif strat == "child":
        print("   💡 Best for: Precise matching + full parent context")
        print("   ⚠️  Watch: Requires parent lookup overhead")
    elif strat == "late_chunk_base":
        print("   💡 Best for: Flexible index + query-time refinement")
        print("   ⚠️  Watch: Base passages may be too large for short docs")

# Overall recommendation
print("\n" + "=" * 80)
print("🎯 RECOMMENDATION")
print("=" * 80)

fastest = summary["Avg_Latency_s"].idxmin()
most_precise = summary["Avg_Precision_5"].idxmax()
best_recall = summary["Avg_Recall_5"].idxmax()
best_relevance = summary["Avg_Answer_Relevance"].idxmax()

print(f"\n   🏆 Overall Winner:    {winner.upper()} (composite {summary.loc[winner, 'Composite_Score']:.3f})")
print(f"   🎯 Most Precise:     {most_precise.upper()} (P@5 = {summary.loc[most_precise, 'Avg_Precision_5']:.3f})")
print(f"   📡 Best Recall:      {best_recall.upper()} (R@5 = {summary.loc[best_recall, 'Avg_Recall_5']:.3f})")
print(f"   ✍️  Best Relevance:   {best_relevance.upper()} (Rel = {summary.loc[best_relevance, 'Avg_Answer_Relevance']:.3f})")
print(f"   ⚡ Fastest:          {fastest.upper()} (Avg {summary.loc[fastest, 'Avg_Latency_s']:.2f}s)")

print("\n" + "=" * 80)

📋 FINAL VERDICT: CHUNKING STRATEGY TRADE-OFFS

🥇 CHILD (Rank #1)
   Composite: 0.513 | P@5: 0.400 | R@5: 0.623 | Rel: 0.515 | Latency: 5.42s
   💡 Best for: Precise matching + full parent context
   ⚠️  Watch: Requires parent lookup overhead

🥈 FIXED (Rank #2)
   Composite: 0.430 | P@5: 0.200 | R@5: 0.553 | Rel: 0.510 | Latency: 4.56s
   💡 Best for: Predictable, uniform retrieval
   ⚠️  Watch: Splits mid-section, ignores document structure

🥉 SLIDING (Rank #3)
   Composite: 0.429 | P@5: 0.260 | R@5: 0.503 | Rel: 0.501 | Latency: 4.86s
   💡 Best for: Maximum coverage, no content gaps
   ⚠️  Watch: 50% redundancy, diluted embeddings

   SEMANTIC (Rank #4)
   Composite: 0.401 | P@5: 0.140 | R@5: 0.478 | Rel: 0.540 | Latency: 4.51s
   💡 Best for: Topic-specific queries (price lookups, feature queries)
   ⚠️  Watch: May miss cross-section answers

   LATE_CHUNK_BASE (Rank #5)
   Composite: 0.379 | P@5: 0.160 | R@5: 0.503 | Rel: 0.450 | Latency: 4.43s
   💡 Best for: Flexible index + query-tim

---
## 2⭐️ CAG Cache Effectiveness

Simulates **100 queries** to measure cache hit rate, latency improvement, and estimated cost savings from avoided API calls.

In [24]:
import random
random.seed(42)

from real_state.application.chat_service import RagService, CAGService, CAGCache
from real_state.config import CACHE_DIR, KNOWN_FAQS, TOP_K_RESULTS

# Initialize services
rag_service = RagService(retriever=db_client, llm=llm, k=TOP_K_RESULTS)

CACHE_DIR.mkdir(parents=True, exist_ok=True)
cache = CAGCache(
    cache_dir=CACHE_DIR,
    embedder=emb,
    similarity_threshold=0.90,
    history_ttl_hours=24
)

cag_service = CAGService(rag_service=rag_service, cache=cache)

# Load FAQs
loaded = cag_service.load_faqs(KNOWN_FAQS)

stats = cache.stats()
print('\u2705 CAG Service initialized')
print(f'\U0001f4cb FAQs loaded: {stats["faq_count"]} ({stats["faq_ready"]} warmed, {stats["faq_pending"]} pending)')
print(f'\U0001f4ca History cached: {stats["history_count"]}')
print(f'\U0001f3af Similarity threshold: {stats["similarity_threshold"]}')
print(f'\U0001f4be Cache size: {stats["cache_size_kb"]:.1f} KB')

✅ CAG Service initialized
📋 FAQs loaded: 22 (22 warmed, 0 pending)
📊 History cached: 3
🎯 Similarity threshold: 0.9
💾 Cache size: 661.8 KB


In [25]:
# ============================================================
# Build 100-query simulation pool
# ============================================================
# Mix: ~40% FAQ variants, ~30% repeated novel, ~30% unique novel

faq_variants = [
    'What types of properties does Prime Group offer?',
    'How much does SKYE BLOSSOM KOTTAWA cost?',
    'What are the available payment plans at PrimeLands?',
    'Which locations does PrimeLands have projects in?',
    'What amenities are included in PrimeLands housing projects?',
    'What kind of properties can I buy from Prime Group?',
    'How much is SKYE BLOSSOM in KOTTAWA?',
    'What payment options does PrimeLands provide?',
    'Where are PrimeLands projects located?',
    'What facilities come with PrimeLands houses?',
    'Does PrimeLands offer bank loan facilities?',
    'How can I contact PrimeLands?',
    'What is the down payment for PrimeLands properties?',
    'Are there land plots near highway entrances?',
    'What is the price range of PrimeLands land projects?',
]

novel_queries = [
    'What is the price of MAJESTINO MORATUWA?',
    'Tell me about GREEN EMBAZY HOMAGAMA facilities',
    'Which projects are in Kadawatha?',
    'Compare apartments in Colombo and Negombo',
    'Land projects in the south of Sri Lanka',
    'Projects near railway stations with prices',
    'Houses available in Thalawathugoda',
    'Lowest price per perch investment property',
    'What is PRESTIGE PLACE KURUNEGALA price?',
    'Tell me about URBAN PARK ANURADHAPURA',
    'Payment plan for CELESTE JA ELA',
    'What facilities does ETERNITY MINUWANGODA have?',
    'Projects in Moratuwa area',
    'Apartment projects with swimming pool',
    'Land near Kandy main road',
]

simulation_queries = []

# Phase 1: 40 FAQ-style queries
for i in range(40):
    q = faq_variants[i % len(faq_variants)]
    simulation_queries.append({'query': q, 'category': 'faq'})

# Phase 2: 30 novel queries (first run = miss, repeats = hit)
for i in range(30):
    q = novel_queries[i % len(novel_queries)]
    simulation_queries.append({'query': q, 'category': 'novel'})

# Phase 3: 30 repeated queries (should be cache hits)
repeated_pool = faq_variants[:8] + novel_queries[:7]
for i in range(30):
    q = repeated_pool[i % len(repeated_pool)]
    simulation_queries.append({'query': q, 'category': 'repeat'})

# Shuffle to simulate realistic user traffic
random.shuffle(simulation_queries)

print(f'\u2705 Built {len(simulation_queries)} simulation queries')
cats = {}
for sq in simulation_queries:
    cats[sq["category"]] = cats.get(sq["category"], 0) + 1
for cat, count in sorted(cats.items()):
    print(f'   {cat}: {count} queries')

✅ Built 100 simulation queries
   faq: 40 queries
   novel: 30 queries
   repeat: 30 queries


In [26]:
# ============================================================
# Run 100-Query Simulation
# ============================================================

print('=' * 80)
print('CAG CACHE EFFECTIVENESS: 100-Query Simulation')
print('=' * 80)

cag_results = []

for i, sq in enumerate(simulation_queries, 1):
    query = sq['query']
    category = sq['category']
    
    if i % 20 == 1:
        print(f'\nProgress: {i-1}/100 completed...')
    
    try:
        result = cag_service.generate(query, use_cache=True, verbose=False)
        
        cag_results.append({
            'query_num': i,
            'query': query,
            'category': category,
            'cache_hit': result.get('cache_hit', False),
            'cache_source': result.get('cache_source', None),
            'latency_s': result.get('generation_time', 0),
            'similarity_score': result.get('similarity_score', 0.0),
        })
        
        status = 'HIT' if result.get('cache_hit') else 'MISS'
        src = result.get('cache_source', '-') or '-'
        if i <= 10 or i % 25 == 0 or not result.get('cache_hit'):
            print(f'   [{i:3d}/100] {status} ({src:>7s}) {result["generation_time"]:.2f}s | {query[:50]}')
    
    except Exception as e:
        print(f'   [{i:3d}/100] Error: {str(e)[:50]}')
        cag_results.append({
            'query_num': i,
            'query': query,
            'category': category,
            'cache_hit': False,
            'cache_source': None,
            'latency_s': 0,
            'similarity_score': 0.0,
        })

print(f'\nSimulation complete: {len(cag_results)} queries processed')

CAG CACHE EFFECTIVENESS: 100-Query Simulation

Progress: 0/100 completed...
   [  1/100] MISS (      -) 6.75s | Which projects are in Kadawatha?
   [  2/100] MISS (      -) 4.85s | Tell me about GREEN EMBAZY HOMAGAMA facilities
   [  3/100] HIT (    faq) 0.48s | How much is SKYE BLOSSOM in KOTTAWA?
   [  4/100] MISS (      -) 9.89s | What facilities come with PrimeLands houses?
   [  5/100] MISS (      -) 4.73s | Payment plan for CELESTE JA ELA
   [  6/100] HIT (history) 0.33s | Payment plan for CELESTE JA ELA
   [  7/100] HIT (    faq) 0.80s | How much does SKYE BLOSSOM KOTTAWA cost?
   [  8/100] HIT (    faq) 0.44s | What types of properties does Prime Group offer?
   [  9/100] HIT (    faq) 0.48s | What types of properties does Prime Group offer?
   [ 10/100] MISS (      -) 5.70s | What is the price of MAJESTINO MORATUWA?
   [ 12/100] MISS (      -) 6.96s | Does PrimeLands offer bank loan facilities?
   [ 16/100] MISS (      -) 6.16s | Projects near railway stations with prices
   [

In [27]:
# ============================================================
# Analysis: Cache Hit Rate, Latency, Cost Savings
# ============================================================

cag_df = pd.DataFrame(cag_results)

total = len(cag_df)
hits = cag_df['cache_hit'].sum()
misses = total - hits
hit_rate = hits / total * 100

# Latency analysis
hit_latencies = cag_df[cag_df['cache_hit'] == True]['latency_s']
miss_latencies = cag_df[cag_df['cache_hit'] == False]['latency_s']

avg_hit_latency = hit_latencies.mean() if len(hit_latencies) > 0 else 0
avg_miss_latency = miss_latencies.mean() if len(miss_latencies) > 0 else 0
speedup = avg_miss_latency / avg_hit_latency if avg_hit_latency > 0 else 0

# By source
faq_hits = len(cag_df[cag_df['cache_source'] == 'faq'])
history_hits = len(cag_df[cag_df['cache_source'] == 'history'])

# By category
cat_stats = cag_df.groupby('category').agg(
    total=('query_num', 'count'),
    hits=('cache_hit', 'sum'),
    avg_latency=('latency_s', 'mean')
).round(3)
cat_stats['hit_rate_%'] = (cat_stats['hits'] / cat_stats['total'] * 100).round(1)

print('=' * 80)
print('CAG CACHE EFFECTIVENESS RESULTS')
print('=' * 80)

print(f'\nOverall Cache Hit Rate: {hit_rate:.1f}% ({hits}/{total})')
print(f'   FAQ hits:     {faq_hits}')
print(f'   History hits: {history_hits}')
print(f'   Misses:       {misses}')

print(f'\nLatency Improvement:')
print(f'   Cache HIT avg:  {avg_hit_latency:.3f}s')
print(f'   Cache MISS avg: {avg_miss_latency:.2f}s')
print(f'   Speedup:        {speedup:.1f}x faster')

print(f'\nBy Query Category:')
print(cat_stats.to_string())

# Cache stats
final_stats = cache.stats()
print(f'\nFinal Cache State:')
print(f'   Total cached: {final_stats["total_cached"]}')
print(f'   FAQs: {final_stats["faq_count"]} ({final_stats["faq_ready"]} warmed)')
print(f'   History: {final_stats["history_count"]}')
print(f'   Cache size: {final_stats["cache_size_kb"]:.1f} KB')

CAG CACHE EFFECTIVENESS RESULTS

Overall Cache Hit Rate: 80.0% (80/100)
   FAQ hits:     39
   History hits: 41
   Misses:       20

Latency Improvement:
   Cache HIT avg:  0.485s
   Cache MISS avg: 6.42s
   Speedup:        13.2x faster

By Query Category:
          total  hits  avg_latency  hit_rate_%
category                                      
faq          40    36        1.212        90.0
novel        30    17        2.916        56.7
repeat       30    27        1.045        90.0

Final Cache State:
   Total cached: 45
   FAQs: 22 (22 warmed)
   History: 23
   Cache size: 1183.4 KB


In [28]:
# ============================================================
# Cost Savings Estimate
# ============================================================

# Pricing assumptions (OpenRouter)
COST_PER_LLM_CALL = 0.002         # ~$0.002 per generation (Gemini Flash)
COST_PER_EMBEDDING = 0.0001        # ~$0.0001 per embedding call
DAILY_QUERIES = 500                # Assumed daily query volume
DAYS_PER_MONTH = 30

monthly_queries = DAILY_QUERIES * DAYS_PER_MONTH

# Without cache: every query = 1 embedding + 1 LLM call
cost_no_cache = monthly_queries * (COST_PER_LLM_CALL + COST_PER_EMBEDDING)

# With cache: hits only need 1 embedding (for similarity lookup), no LLM
cache_hit_ratio = hit_rate / 100
monthly_hits = int(monthly_queries * cache_hit_ratio)
monthly_misses = monthly_queries - monthly_hits

cost_with_cache = (
    monthly_hits * COST_PER_EMBEDDING +           # Hits: embedding only
    monthly_misses * (COST_PER_LLM_CALL + COST_PER_EMBEDDING)  # Misses: full pipeline
)

savings = cost_no_cache - cost_with_cache
savings_pct = (savings / cost_no_cache * 100) if cost_no_cache > 0 else 0

# Time savings
time_no_cache = monthly_queries * avg_miss_latency
time_with_cache = monthly_hits * avg_hit_latency + monthly_misses * avg_miss_latency
time_saved_hours = (time_no_cache - time_with_cache) / 3600

print('=' * 80)
print('MONTHLY COST SAVINGS ESTIMATE')
print('=' * 80)

print(f'\nAssumptions:')
print(f'   Daily queries:         {DAILY_QUERIES:,}')
print(f'   Monthly queries:       {monthly_queries:,}')
print(f'   Cache hit rate:        {hit_rate:.1f}%')
print(f'   Cost per LLM call:     ${COST_PER_LLM_CALL}')
print(f'   Cost per embedding:    ${COST_PER_EMBEDDING}')

print(f'\nCost Comparison:')
print(f'   Without cache:  ${cost_no_cache:,.2f}/month')
print(f'   With cache:     ${cost_with_cache:,.2f}/month')
print(f'   Monthly savings: ${savings:,.2f} ({savings_pct:.1f}%)')
print(f'   Annual savings:  ${savings * 12:,.2f}')

print(f'\nLatency Savings:')
print(f'   Without cache:  {time_no_cache/3600:.1f} hours/month of wait time')
print(f'   With cache:     {time_with_cache/3600:.1f} hours/month')
print(f'   Time saved:     {time_saved_hours:.1f} hours/month')

print(f'\nAPI Calls Avoided: {monthly_hits:,} LLM calls/month')

print('\n' + '=' * 80)
print('SUMMARY')
print('=' * 80)
print(f'   Cache hit rate:    {hit_rate:.1f}%')
print(f'   Latency speedup:  {speedup:.1f}x')
print(f'   Cost reduction:   {savings_pct:.1f}%')
print(f'   Monthly savings:  ${savings:,.2f}')
print('=' * 80)

MONTHLY COST SAVINGS ESTIMATE

Assumptions:
   Daily queries:         500
   Monthly queries:       15,000
   Cache hit rate:        80.0%
   Cost per LLM call:     $0.002
   Cost per embedding:    $0.0001

Cost Comparison:
   Without cache:  $31.50/month
   With cache:     $7.50/month
   Monthly savings: $24.00 (76.2%)
   Annual savings:  $288.00

Latency Savings:
   Without cache:  26.8 hours/month of wait time
   With cache:     7.0 hours/month
   Time saved:     19.8 hours/month

API Calls Avoided: 12,000 LLM calls/month

SUMMARY
   Cache hit rate:    80.0%
   Latency speedup:  13.2x
   Cost reduction:   76.2%
   Monthly savings:  $24.00


In [33]:
# Save CAG simulation results as structured JSON
import json as _json

cag_stats = {
    'simulation': {
        'total_queries': int(total),
        'cache_hits': int(hits),
        'cache_misses': int(misses),
        'hit_rate_pct': round(hit_rate, 1),
        'faq_hits': int(faq_hits),
        'history_hits': int(history_hits),
    },
    'latency': {
        'avg_hit_latency_s': round(float(avg_hit_latency), 3),
        'avg_miss_latency_s': round(float(avg_miss_latency), 3),
        'speedup_factor': round(float(speedup), 1),
    },
    'cost_savings': {
        'daily_queries': DAILY_QUERIES,
        'monthly_queries': monthly_queries,
        'cost_per_llm_call': COST_PER_LLM_CALL,
        'cost_per_embedding': COST_PER_EMBEDDING,
        'monthly_cost_without_cache': round(float(cost_no_cache), 2),
        'monthly_cost_with_cache': round(float(cost_with_cache), 2),
        'monthly_savings_usd': round(float(savings), 2),
        'savings_pct': round(float(savings_pct), 1),
        'annual_savings_usd': round(float(savings * 12), 2),
        'api_calls_avoided_monthly': monthly_hits,
        'time_saved_hours_monthly': round(float(time_saved_hours), 1),
    },
    'cache_state': {
        'total_cached': final_stats['total_cached'],
        'faq_count': final_stats['faq_count'],
        'faq_ready': final_stats['faq_ready'],
        'history_count': final_stats['history_count'],
        'cache_size_kb': round(final_stats['cache_size_kb'], 1),
    },
    'per_category': {
        cat: {
            'total': int(row['total']),
            'hits': int(row['hits']),
            'hit_rate_pct': float(row['hit_rate_%']),
            'avg_latency_s': float(row['avg_latency']),
        }
        for cat, row in cat_stats.iterrows()
    },
    'raw_results': cag_df.to_dict(orient='records'),
}

cag_json = CRAWL_OUT_DIR / 'cag_stats.json'
with open(cag_json, 'w') as f:
    _json.dump(cag_stats, f, indent=2, default=str)
print(f'\u2705 CAG stats saved to: {cag_json}')
print(f'   Keys: {list(cag_stats.keys())}')

✅ CAG stats saved to: d:\ai\bootcamp\mini-project-02\real_state\data\cag_stats.json
   Keys: ['simulation', 'latency', 'cost_savings', 'cache_state', 'per_category', 'raw_results']


---
## 3️⃣ CRAG Correction Impact

Compares **RAG vs CRAG** on 20 queries. Tracks correction frequency, confidence improvement, and answer quality gains.

In [34]:
# ============================================================
# CRAG Correction Impact: RAG vs CRAG on 20 Queries
# ============================================================

from real_state.application.chat_service import CRAGService, RagService
from real_state.config import CRAG_CONFIDENCE_THRESHOLD, TOP_K_RESULTS

# Initialize RAG service (standard)
rag_svc = RagService(retriever=db_client, llm=llm, k=TOP_K_RESULTS)

# Initialize CRAG service (corrective)
crag_svc = CRAGService(retriever=db_client, llm=llm)

print(f'✅ RAG Service: k={TOP_K_RESULTS}')
print(f'✅ CRAG Service: initial_k={crag_svc.initial_k}, expanded_k={crag_svc.expanded_k}')
print(f'🎯 Confidence threshold: {CRAG_CONFIDENCE_THRESHOLD}')

✅ RAG Service: k=4
✅ CRAG Service: initial_k=4, expanded_k=8
🎯 Confidence threshold: 0.6


In [35]:
# ============================================================
# 20 Test Queries for RAG vs CRAG Comparison
# ============================================================
# Mix of easy (high confidence) and hard (low confidence) queries

crag_test_queries = [
    # --- Easy / Direct (expect high confidence) ---
    {'query': 'What is the price of MAJESTINO MORATUWA?', 'type': 'Direct Price', 'expected': '1,050,000 LKR'},
    {'query': 'How much does SKYE BLOSSOM KOTTAWA cost?', 'type': 'Direct Price', 'expected': '50,000,000 LKR'},
    {'query': 'What amenities does GREEN EMBAZY HOMAGAMA have?', 'type': 'Feature Query', 'expected': 'Facilities list'},
    {'query': 'What is the payment plan for EVARA NAWALAPITIYA?', 'type': 'Payment Plan', 'expected': 'Down payment + installments'},
    {'query': 'What is PRESTIGE PLACE KURUNEGALA price?', 'type': 'Direct Price', 'expected': 'Price info'},
    
    # --- Medium (multi-document, some ambiguity) ---
    {'query': 'Which PrimeLands projects are available in Kadawatha?', 'type': 'Location Filter', 'expected': 'Multiple Kadawatha projects'},
    {'query': 'What houses are available in Thalawathugoda?', 'type': 'Category + Location', 'expected': 'CEST LA VIE, CLOVER'},
    {'query': 'Projects in Moratuwa area with prices', 'type': 'Location + Price', 'expected': 'MAJESTINO, ZILLION'},
    {'query': 'What facilities does ETERNITY MINUWANGODA have?', 'type': 'Feature Query', 'expected': 'Facilities list'},
    {'query': 'Tell me about URBAN PARK ANURADHAPURA', 'type': 'Project Info', 'expected': 'Project details'},
    
    # --- Hard (vague, multi-part, abstract) ---
    {'query': 'Compare apartment prices in Colombo vs Negombo', 'type': 'Comparison', 'expected': 'THE SEASONS vs VENEZIA, J ADORE'},
    {'query': 'Tell me about land projects in the south', 'type': 'Vague Regional', 'expected': 'Southern projects'},
    {'query': 'Which projects are close to railway stations?', 'type': 'Proximity', 'expected': 'Projects near railways'},
    {'query': 'Which property has the lowest price per perch for investment?', 'type': 'Investment', 'expected': 'Lowest per-perch prices'},
    {'query': 'Land near Kandy main road with good access', 'type': 'Vague + Feature', 'expected': 'Kandy area projects'},
    {'query': 'Apartment projects with swimming pool facilities', 'type': 'Amenity Filter', 'expected': 'Apartments with pools'},
    {'query': 'What are the best investment properties under 5 million?', 'type': 'Budget + Investment', 'expected': 'Low-cost investment options'},
    {'query': 'PrimeLands projects near Colombo with installment plans', 'type': 'Multi-criteria', 'expected': 'Colombo area + payment plans'},
    {'query': 'Which land projects offer the largest plot sizes?', 'type': 'Feature Comparison', 'expected': 'Large plot projects'},
    {'query': 'Are there any beachfront or ocean view properties?', 'type': 'Feature Filter', 'expected': 'Coastal projects'},
]

print(f'✅ {len(crag_test_queries)} queries prepared for RAG vs CRAG comparison')
for i, q in enumerate(crag_test_queries, 1):
    print(f'   {i:2d}. [{q["type"]:20s}] {q["query"][:55]}')

✅ 20 queries prepared for RAG vs CRAG comparison
    1. [Direct Price        ] What is the price of MAJESTINO MORATUWA?
    2. [Direct Price        ] How much does SKYE BLOSSOM KOTTAWA cost?
    3. [Feature Query       ] What amenities does GREEN EMBAZY HOMAGAMA have?
    4. [Payment Plan        ] What is the payment plan for EVARA NAWALAPITIYA?
    5. [Direct Price        ] What is PRESTIGE PLACE KURUNEGALA price?
    6. [Location Filter     ] Which PrimeLands projects are available in Kadawatha?
    7. [Category + Location ] What houses are available in Thalawathugoda?
    8. [Location + Price    ] Projects in Moratuwa area with prices
    9. [Feature Query       ] What facilities does ETERNITY MINUWANGODA have?
   10. [Project Info        ] Tell me about URBAN PARK ANURADHAPURA
   11. [Comparison          ] Compare apartment prices in Colombo vs Negombo
   12. [Vague Regional      ] Tell me about land projects in the south
   13. [Proximity           ] Which projects are close to ra

In [36]:
# ============================================================
# Run RAG vs CRAG Comparison: 20 Queries
# ============================================================

print('=' * 80)
print('🔬 CRAG CORRECTION IMPACT: RAG vs CRAG on 20 Queries')
print('=' * 80)

comparison_results = []

for i, tq in enumerate(crag_test_queries, 1):
    query = tq['query']
    qtype = tq['type']
    expected = tq['expected']
    
    print(f'\n📝 Query {i:2d}/20: {query[:55]}...')
    
    # --- Run RAG ---
    try:
        t_rag_start = time.time()
        rag_result = rag_svc.generate(query)
        t_rag = time.time() - t_rag_start
        
        rag_answer = rag_result['answer']
        rag_urls = rag_result.get('evidence_urls', [])
        rag_latency = t_rag
        rag_relevance = answer_relevance(rag_answer, expected, query)
    except Exception as e:
        print(f'   RAG ❌ Error: {str(e)[:50]}')
        rag_answer = f'ERROR: {str(e)[:80]}'
        rag_urls = []
        rag_latency = 0
        rag_relevance = 0.0
    
    # --- Run CRAG ---
    try:
        t_crag_start = time.time()
        crag_result = crag_svc.generate(query, verbose=False)
        t_crag = time.time() - t_crag_start
        
        crag_answer = crag_result['answer']
        crag_urls = crag_result.get('evidence_urls', [])
        crag_latency = t_crag
        crag_relevance = answer_relevance(crag_answer, expected, query)
        conf_initial = crag_result.get('confidence_initial', 0)
        conf_final = crag_result.get('confidence_final', 0)
        correction = crag_result.get('correction_applied', False)
        docs_used = crag_result.get('docs_used', 0)
    except Exception as e:
        print(f'   CRAG ❌ Error: {str(e)[:50]}')
        crag_answer = f'ERROR: {str(e)[:80]}'
        crag_urls = []
        crag_latency = 0
        crag_relevance = 0.0
        conf_initial = 0
        conf_final = 0
        correction = False
        docs_used = 0
    
    # Log result
    status = '🔄 CORRECTED' if correction else '✅ DIRECT'
    conf_delta = conf_final - conf_initial
    rel_delta = crag_relevance - rag_relevance
    
    print(f'   RAG:  Rel={rag_relevance:.2f}  ⏱️{rag_latency:.1f}s')
    print(f'   CRAG: Rel={crag_relevance:.2f}  ⏱️{crag_latency:.1f}s  Conf:{conf_initial:.2f}→{conf_final:.2f}  {status}')
    
    if rel_delta > 0:
        print(f'   📈 CRAG wins by +{rel_delta:.2f} relevance')
    elif rel_delta < 0:
        print(f'   📉 RAG wins by +{abs(rel_delta):.2f} relevance')
    else:
        print(f'   🤝 Tie')
    
    comparison_results.append({
        'query_id': i,
        'query': query,
        'query_type': qtype,
        'rag_relevance': round(rag_relevance, 3),
        'crag_relevance': round(crag_relevance, 3),
        'relevance_delta': round(rel_delta, 3),
        'rag_latency_s': round(rag_latency, 2),
        'crag_latency_s': round(crag_latency, 2),
        'confidence_initial': round(conf_initial, 3),
        'confidence_final': round(conf_final, 3),
        'confidence_delta': round(conf_delta, 3),
        'correction_applied': correction,
        'crag_docs_used': docs_used,
        'rag_answer_preview': rag_answer[:100],
        'crag_answer_preview': crag_answer[:100],
    })

print(f'\n✅ Comparison complete: {len(comparison_results)} queries')

🔬 CRAG CORRECTION IMPACT: RAG vs CRAG on 20 Queries

📝 Query  1/20: What is the price of MAJESTINO MORATUWA?...
   RAG:  Rel=0.90  ⏱️9.2s
   CRAG: Rel=0.90  ⏱️6.8s  Conf:0.69→0.69  ✅ DIRECT
   🤝 Tie

📝 Query  2/20: How much does SKYE BLOSSOM KOTTAWA cost?...
   RAG:  Rel=0.83  ⏱️3.4s
   CRAG: Rel=0.83  ⏱️3.2s  Conf:0.67→0.67  ✅ DIRECT
   🤝 Tie

📝 Query  3/20: What amenities does GREEN EMBAZY HOMAGAMA have?...
   RAG:  Rel=0.35  ⏱️3.6s
   CRAG: Rel=0.40  ⏱️4.2s  Conf:0.56→0.51  🔄 CORRECTED
   📈 CRAG wins by +0.05 relevance

📝 Query  4/20: What is the payment plan for EVARA NAWALAPITIYA?...
   RAG:  Rel=0.68  ⏱️3.4s
   CRAG: Rel=0.68  ⏱️4.0s  Conf:0.79→0.79  ✅ DIRECT
   🤝 Tie

📝 Query  5/20: What is PRESTIGE PLACE KURUNEGALA price?...
   RAG:  Rel=0.68  ⏱️4.0s
   CRAG: Rel=0.68  ⏱️3.7s  Conf:0.88→0.88  ✅ DIRECT
   🤝 Tie

📝 Query  6/20: Which PrimeLands projects are available in Kadawatha?...
   RAG:  Rel=0.65  ⏱️4.7s
   CRAG: Rel=0.59  ⏱️5.4s  Conf:0.45→0.53  🔄 CORRECTED
   📉 RAG wins by

In [37]:
# ============================================================
# CRAG Correction Impact Analysis
# ============================================================

crag_comp_df = pd.DataFrame(comparison_results)

# --- 1. Correction Frequency ---
total_queries = len(crag_comp_df)
corrections = crag_comp_df['correction_applied'].sum()
correction_rate = corrections / total_queries * 100

# --- 2. Confidence Improvement ---
corrected_df = crag_comp_df[crag_comp_df['correction_applied'] == True]
direct_df = crag_comp_df[crag_comp_df['correction_applied'] == False]

avg_conf_initial = crag_comp_df['confidence_initial'].mean()
avg_conf_final = crag_comp_df['confidence_final'].mean()
avg_conf_improvement = crag_comp_df['confidence_delta'].mean()

if len(corrected_df) > 0:
    avg_corrected_improvement = corrected_df['confidence_delta'].mean()
    max_improvement = corrected_df['confidence_delta'].max()
else:
    avg_corrected_improvement = 0
    max_improvement = 0

# --- 3. Answer Quality Gains ---
avg_rag_relevance = crag_comp_df['rag_relevance'].mean()
avg_crag_relevance = crag_comp_df['crag_relevance'].mean()
avg_rel_delta = crag_comp_df['relevance_delta'].mean()

crag_wins = (crag_comp_df['relevance_delta'] > 0).sum()
rag_wins = (crag_comp_df['relevance_delta'] < 0).sum()
ties = (crag_comp_df['relevance_delta'] == 0).sum()

# --- 4. Latency Comparison ---
avg_rag_latency = crag_comp_df['rag_latency_s'].mean()
avg_crag_latency = crag_comp_df['crag_latency_s'].mean()
latency_overhead = avg_crag_latency - avg_rag_latency

print('=' * 80)
print('🔬 CRAG CORRECTION IMPACT RESULTS')
print('=' * 80)

print(f'\n📊 Correction Frequency:')
print(f'   Total queries:        {total_queries}')
print(f'   Corrections applied:  {corrections} ({correction_rate:.1f}%)')
print(f'   Direct (no correction): {total_queries - corrections} ({100 - correction_rate:.1f}%)')

print(f'\n📈 Confidence Improvement:')
print(f'   Avg initial confidence:  {avg_conf_initial:.3f}')
print(f'   Avg final confidence:    {avg_conf_final:.3f}')
print(f'   Avg improvement (all):   +{avg_conf_improvement:.3f}')
if len(corrected_df) > 0:
    print(f'   Avg improvement (corrected only): +{avg_corrected_improvement:.3f}')
    print(f'   Max improvement:         +{max_improvement:.3f}')

print(f'\n✍️  Answer Quality (Relevance Score):')
print(f'   RAG avg relevance:   {avg_rag_relevance:.3f}')
print(f'   CRAG avg relevance:  {avg_crag_relevance:.3f}')
print(f'   Avg delta:           {avg_rel_delta:+.3f}')
print(f'   CRAG wins: {crag_wins}  |  RAG wins: {rag_wins}  |  Ties: {ties}')

print(f'\n⏱️  Latency Comparison:')
print(f'   RAG avg:    {avg_rag_latency:.2f}s')
print(f'   CRAG avg:   {avg_crag_latency:.2f}s')
print(f'   Overhead:   {latency_overhead:+.2f}s')
if len(corrected_df) > 0:
    print(f'   Corrected avg latency: {corrected_df["crag_latency_s"].mean():.2f}s')
    print(f'   Direct avg latency:    {direct_df["crag_latency_s"].mean():.2f}s')

print('\n' + '=' * 80)

🔬 CRAG CORRECTION IMPACT RESULTS

📊 Correction Frequency:
   Total queries:        20
   Corrections applied:  9 (45.0%)
   Direct (no correction): 11 (55.0%)

📈 Confidence Improvement:
   Avg initial confidence:  0.618
   Avg final confidence:    0.641
   Avg improvement (all):   +0.022
   Avg improvement (corrected only): +0.049
   Max improvement:         +0.093

✍️  Answer Quality (Relevance Score):
   RAG avg relevance:   0.523
   CRAG avg relevance:  0.500
   Avg delta:           -0.023
   CRAG wins: 3  |  RAG wins: 6  |  Ties: 11

⏱️  Latency Comparison:
   RAG avg:    4.76s
   CRAG avg:   5.04s
   Overhead:   +0.29s
   Corrected avg latency: 5.87s
   Direct avg latency:    4.37s



In [38]:
# ============================================================
# Per-Query Breakdown Table
# ============================================================

print('📋 Per-Query RAG vs CRAG Comparison')
print('=' * 100)
print(f'{"Q#":>3s} {"Type":>20s} {"RAG_Rel":>8s} {"CRAG_Rel":>9s} {"Δ_Rel":>7s} {"Conf_i":>7s} {"Conf_f":>7s} {"Δ_Conf":>7s} {"Corrected":>10s}')
print('-' * 100)

for _, row in crag_comp_df.iterrows():
    corr_flag = '🔄 YES' if row['correction_applied'] else '  —'
    rel_sign = '+' if row['relevance_delta'] > 0 else ''
    conf_sign = '+' if row['confidence_delta'] > 0 else ''
    print(f'{row["query_id"]:3d} {row["query_type"]:>20s} '
          f'{row["rag_relevance"]:8.3f} {row["crag_relevance"]:9.3f} '
          f'{rel_sign}{row["relevance_delta"]:6.3f} '
          f'{row["confidence_initial"]:7.3f} {row["confidence_final"]:7.3f} '
          f'{conf_sign}{row["confidence_delta"]:6.3f} '
          f'{corr_flag:>10s}')

print('-' * 100)
print(f'{"AVG":>3s} {"":>20s} '
      f'{avg_rag_relevance:8.3f} {avg_crag_relevance:9.3f} '
      f'{avg_rel_delta:+7.3f} '
      f'{avg_conf_initial:7.3f} {avg_conf_final:7.3f} '
      f'{avg_conf_improvement:+7.3f} '
      f'{correction_rate:8.1f}%')
print('=' * 100)

# By query type breakdown
print('\n📊 By Query Type:')
print('=' * 80)
type_stats = crag_comp_df.groupby('query_type').agg(
    n=('query_id', 'count'),
    rag_rel=('rag_relevance', 'mean'),
    crag_rel=('crag_relevance', 'mean'),
    delta_rel=('relevance_delta', 'mean'),
    corrections=('correction_applied', 'sum'),
    avg_conf_delta=('confidence_delta', 'mean'),
).round(3)
type_stats['correction_rate_%'] = (type_stats['corrections'] / type_stats['n'] * 100).round(1)
print(type_stats.to_string())

📋 Per-Query RAG vs CRAG Comparison
 Q#                 Type  RAG_Rel  CRAG_Rel   Δ_Rel  Conf_i  Conf_f  Δ_Conf  Corrected
----------------------------------------------------------------------------------------------------
  1         Direct Price    0.900     0.900  0.000   0.688   0.688  0.000          —
  2         Direct Price    0.829     0.829  0.000   0.674   0.674  0.000          —
  3        Feature Query    0.350     0.400 + 0.050   0.559   0.514 -0.045      🔄 YES
  4         Payment Plan    0.675     0.675  0.000   0.788   0.788  0.000          —
  5         Direct Price    0.675     0.675  0.000   0.877   0.877  0.000          —
  6      Location Filter    0.653     0.593 -0.060   0.451   0.531 + 0.080      🔄 YES
  7  Category + Location    0.400     0.200 -0.200   0.503   0.583 + 0.080      🔄 YES
  8     Location + Price    0.630     0.630  0.000   0.691   0.691  0.000          —
  9        Feature Query    0.320     0.630 + 0.310   0.503   0.533 + 0.030      🔄 YES
 10    

In [39]:
# ============================================================
# CRAG Impact Verdict
# ============================================================

print('=' * 80)
print('🏆 CRAG CORRECTION IMPACT VERDICT')
print('=' * 80)

print(f'\n📊 Key Findings:')
print(f'   1. Correction Rate: {correction_rate:.1f}% of queries triggered corrective retrieval')
print(f'   2. Confidence Gain: Average +{avg_conf_improvement:.3f} ({avg_conf_improvement*100:.1f}%) improvement')
if len(corrected_df) > 0:
    print(f'      — Corrected queries improved by +{avg_corrected_improvement:.3f}')
print(f'   3. Answer Quality:  CRAG avg={avg_crag_relevance:.3f} vs RAG avg={avg_rag_relevance:.3f} ({avg_rel_delta:+.3f})')
print(f'   4. Win Rate:        CRAG wins {crag_wins}/{total_queries} ({crag_wins/total_queries*100:.0f}%), '
      f'RAG wins {rag_wins}/{total_queries} ({rag_wins/total_queries*100:.0f}%), '
      f'Ties {ties}/{total_queries}')
print(f'   5. Latency Cost:    +{latency_overhead:.2f}s avg overhead per query')

print(f'\n💡 When CRAG Helps Most:')
if len(corrected_df) > 0:
    best_improvement = corrected_df.loc[corrected_df['relevance_delta'].idxmax()]
    print(f'   Best improvement: Q{int(best_improvement["query_id"])} [{best_improvement["query_type"]}]')
    print(f'     "{best_improvement["query"]}"')
    print(f'     Relevance: {best_improvement["rag_relevance"]:.3f} → {best_improvement["crag_relevance"]:.3f} (+{best_improvement["relevance_delta"]:.3f})')
    print(f'     Confidence: {best_improvement["confidence_initial"]:.3f} → {best_improvement["confidence_final"]:.3f}')

print(f'\n⚠️  When CRAG Doesn\'t Help:')
if len(direct_df) > 0:
    print(f'   {len(direct_df)} queries had sufficient initial confidence (no correction needed)')
    print(f'   These queries show CRAG = RAG performance (no overhead penalty)')

# Overall verdict
print(f'\n' + '=' * 80)
print(f'🎯 RECOMMENDATION')
print(f'=' * 80)
if avg_rel_delta > 0:
    print(f'\n   ✅ CRAG provides measurable quality improvement (+{avg_rel_delta:.3f} avg relevance)')
    print(f'   ✅ Self-correction helps {correction_rate:.0f}% of queries')
    print(f'   ⚠️  Latency overhead of {latency_overhead:.1f}s may be acceptable for quality gains')
    print(f'   💡 RECOMMENDATION: Use CRAG for production — quality gains outweigh latency cost')
else:
    print(f'\n   ℹ️  CRAG shows marginal improvement ({avg_rel_delta:+.3f} avg relevance)')
    print(f'   ⚠️  Latency overhead of {latency_overhead:.1f}s per query')
    print(f'   💡 RECOMMENDATION: Use CRAG selectively for complex/vague queries')

print('\n' + '=' * 80)

🏆 CRAG CORRECTION IMPACT VERDICT

📊 Key Findings:
   1. Correction Rate: 45.0% of queries triggered corrective retrieval
   2. Confidence Gain: Average +0.022 (2.2%) improvement
      — Corrected queries improved by +0.049
   3. Answer Quality:  CRAG avg=0.500 vs RAG avg=0.523 (-0.023)
   4. Win Rate:        CRAG wins 3/20 (15%), RAG wins 6/20 (30%), Ties 11/20
   5. Latency Cost:    +0.29s avg overhead per query

💡 When CRAG Helps Most:
   Best improvement: Q9 [Feature Query]
     "What facilities does ETERNITY MINUWANGODA have?"
     Relevance: 0.320 → 0.630 (+0.310)
     Confidence: 0.503 → 0.533

⚠️  When CRAG Doesn't Help:
   11 queries had sufficient initial confidence (no correction needed)
   These queries show CRAG = RAG performance (no overhead penalty)

🎯 RECOMMENDATION

   ℹ️  CRAG shows marginal improvement (-0.023 avg relevance)
   ⚠️  Latency overhead of 0.3s per query
   💡 RECOMMENDATION: Use CRAG selectively for complex/vague queries



In [40]:
# ============================================================
# Save CRAG Comparison Results
# ============================================================

# Save to CSV
crag_csv_path = CRAWL_OUT_DIR / 'crag_impact.csv'
crag_comp_df.to_csv(crag_csv_path, index=False)
print(f'✅ CRAG comparison saved to: {crag_csv_path}')
print(f'   {len(crag_comp_df)} rows × {len(crag_comp_df.columns)} columns')

# Save summary JSON
crag_summary = {
    'correction_frequency': {
        'total_queries': total_queries,
        'corrections_applied': int(corrections),
        'correction_rate_pct': round(correction_rate, 1),
    },
    'confidence_improvement': {
        'avg_initial_confidence': round(float(avg_conf_initial), 3),
        'avg_final_confidence': round(float(avg_conf_final), 3),
        'avg_improvement': round(float(avg_conf_improvement), 3),
        'avg_improvement_corrected_only': round(float(avg_corrected_improvement), 3),
        'max_improvement': round(float(max_improvement), 3),
    },
    'answer_quality': {
        'rag_avg_relevance': round(float(avg_rag_relevance), 3),
        'crag_avg_relevance': round(float(avg_crag_relevance), 3),
        'avg_relevance_delta': round(float(avg_rel_delta), 3),
        'crag_wins': int(crag_wins),
        'rag_wins': int(rag_wins),
        'ties': int(ties),
    },
    'latency': {
        'rag_avg_s': round(float(avg_rag_latency), 2),
        'crag_avg_s': round(float(avg_crag_latency), 2),
        'overhead_s': round(float(latency_overhead), 2),
    },
}

crag_json_path = CRAWL_OUT_DIR / 'crag_impact_summary.json'
with open(crag_json_path, 'w') as f:
    json.dump(crag_summary, f, indent=2)
print(f'✅ CRAG summary saved to: {crag_json_path}')

✅ CRAG comparison saved to: d:\ai\bootcamp\mini-project-02\real_state\data\crag_impact.csv
   20 rows × 15 columns
✅ CRAG summary saved to: d:\ai\bootcamp\mini-project-02\real_state\data\crag_impact_summary.json
